## Random File Upload with Cloudflare R2 Persistence

A compact end-to-end example that creates a random binary file and stores it in Cloudflare R2 using LAILA.

### What this notebook does
- Create a random binary file in the current directory (`./random_blob.bin`)
- Read file bytes and wrap them as a LAILA entry (`laila.constant`) to get a global ID
- Persist the entry to Cloudflare via `CloudflarePool` (`memorize`)
- Retrieve the entry from Cloudflare (`remember`) and validate integrity
- Remove the stored object (`forget`) for cleanup

### Credentials
This notebook reads Cloudflare credentials from:
`laila.args`


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import laila

In [ ]:
from laila.pool import CloudflarePool

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own Cloudflare R2 values before running.
laila.args.CLOUDFLARE_ACCOUNT_ID = "your-account-id"
laila.args.CLOUDFLARE_ACCESS_KEY_ID = "your-access-key-id"
laila.args.CLOUDFLARE_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.CLOUDFLARE_BUCKET_NAME = "your-bucket-name"

cloudflare_pool = CloudflarePool(
    account_id=laila.args.CLOUDFLARE_ACCOUNT_ID,
    access_key_id=laila.args.CLOUDFLARE_ACCESS_KEY_ID,
    secret_access_key=laila.args.CLOUDFLARE_SECRET_ACCESS_KEY,
    bucket_name=laila.args.CLOUDFLARE_BUCKET_NAME,
)

In [ ]:
laila.add_pool(cloudflare_pool, pool_nickname="file_backup_cf_pool")

In [ ]:
# Create a random binary file in the current directory (.)
from pathlib import Path
import os

file_path = Path("./random_blob.dev.bin")
file_size_bytes = 16 * 1024 * 1024 # 16 MB

with open(file_path, "wb") as f:
    f.write(os.urandom(file_size_bytes))

print(f"Created: {file_path.resolve()} ({file_size_bytes} bytes)")

In [ ]:
#read the file into a buffer
with open(file_path, "rb") as f:
    buffer = f.read()
#wrap the buffer in a constant to get a global id
wrapped_buffer = laila.constant(data=buffer)
print (wrapped_buffer.global_id)
#store the result in Cloudflare
future_memorize = laila.memorize(wrapped_buffer, pool_nickname="file_backup_cf_pool")
print ("Memorizing to cloudflare...")
future_memorize.wait()
print ("Memorized.")

In [ ]:
#remembering the file
future_remember = laila.remember("LAILA:ENTRY:GLOBAL_ID:1cab2b9b-14bb-4c32-8c06-d14ed6668d30", pool_nickname="file_backup_cf_pool")
future_remember.wait()
print ("Recovered.")


In [ ]:
assert future_remember.result.data == wrapped_buffer.data
print ("Data matches.")